# Cross-modality DEG rank comparison — CosMx vs scRNA-seq

**Goal:** Identify genes concordantly enriched in MHC II–high or MHC II–low malignant
cells across CosMx spatial transcriptomics and Salcher scRNA-seq. Wilcoxon rank-sum
scores are computed independently in each modality and genes are ranked by enrichment
score. Concordance across tiered percentile cutoffs (0.5%, 1%, 2.5%, 5%) is
visualized as rank-scatter plots, with shared genes highlighted by shape and color.

**Input:** `epithelial.h5ad` (CosMx epithelial cells), `cancer_cells_mhc2_classification.h5ad`
(Salcher malignant cells with MHC2_clustering labels).

**Output:** Figure 3A (MHC II–low concordance), Figure 3B (MHC II–high concordance);
ranking CSVs saved to `outputs/tables/figure3/`.

## Setup

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import anndata as ad
import scanpy as sc

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

table_dir = repo_root / cfg['outputs']['tables']
fig_dir   = repo_root / cfg['outputs']['figures']

table_out = table_dir / 'figure3'
fig_out   = fig_dir   / 'figure3'
supp_out  = fig_dir   / 'figure3-supp'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Load data

CosMx epithelial cells and Salcher scRNA-seq malignant cells, both with
MHC II patient classification labels applied in upstream notebooks.

In [ ]:
epithelial_path    = repo_root / cfg['outputs']['processed'] / 'epithelial.h5ad'
cancer_cells_path  = Path(cfg['datasets']['luad_mhc2_classified']['processed'])
patient_class_path = Path(cfg['datasets']['patient_classifications']['raw'])

# load CosMx epithelial cells
cosmx_tumor_data = ad.read_h5ad(epithelial_path)
print(f"CosMx epithelial cells: {cosmx_tumor_data.n_obs:,} × {cosmx_tumor_data.n_vars} genes")

# merge patient classifications — not stored in epithelial.h5ad
patient_classifications = pd.read_csv(patient_class_path, index_col=0)
patient_classifications.loc['tonsil'] = ['Tonsil', 'Tonsil', 'Tonsil']
patient_classifications['patient'] = patient_classifications.index.astype(str)
cosmx_tumor_data.obs['patient'] = cosmx_tumor_data.obs['patient'].astype(str)
cosmx_tumor_data.obs = pd.merge(
    cosmx_tumor_data.obs, patient_classifications, on='patient', how='left'
)

# subset to MHC II positive and negative only
cosmx_tumor_data = cosmx_tumor_data[
    cosmx_tumor_data.obs['patient classification'].isin(['class II positive', 'class II negative'])
].copy()
print(f"After subsetting to pos/neg: {cosmx_tumor_data.n_obs:,} cells")
print(cosmx_tumor_data.obs['patient classification'].value_counts())

# load scRNA malignant cells
sc_tumor_data = sc.read_h5ad(cancer_cells_path)
print(f"\nscRNA malignant cells: {sc_tumor_data.n_obs:,} × {sc_tumor_data.n_vars} genes")
print(sc_tumor_data.obs['MHC2_clustering'].value_counts())

## Wilcoxon rank-sum scoring

Genes are ranked by Wilcoxon score in each modality independently, comparing
MHC II–negative vs MHC II–positive (CosMx) and MHC class II Low vs High (scRNA).
Positive scores indicate enrichment in the MHC II–low group.

### cosmx first

In [ ]:
cosmx_rank_path = table_out / 'cosmx_mhc2_deg_rankings.csv'

if cosmx_rank_path.exists():
    cosmx_rank_df = pd.read_csv(cosmx_rank_path)
    print(f"Loaded CosMx rankings from cache ({len(cosmx_rank_df):,} genes)")
else:
    sc.tl.rank_genes_groups(
        cosmx_tumor_data,
        groupby='patient classification',
        groups=['class II negative'],
        reference='class II positive',
        method='wilcoxon',
        key_added='rank_genes_groups',
    )
    cosmx_rank_df = sc.get.rank_genes_groups_df(
        cosmx_tumor_data, group='class II negative', key='rank_genes_groups'
    )
    if 'feature_name' in cosmx_tumor_data.var.columns:
        gene_to_feature = cosmx_tumor_data.var['feature_name'].to_dict()
        cosmx_rank_df['gene_label'] = cosmx_rank_df['names'].map(gene_to_feature).fillna(cosmx_rank_df['names'])
    else:
        cosmx_rank_df['gene_label'] = cosmx_rank_df['names']

    cosmx_rank_df = cosmx_rank_df.sort_values('scores', ascending=False).reset_index(drop=True)
    cosmx_rank_df.to_csv(cosmx_rank_path, index=False)
    print(f"Computed and saved CosMx rankings ({len(cosmx_rank_df):,} genes)")

### salcher next

#### Note that rankings these throws a warning about the data being raw and not be log scaled
#### Here we can see the data is already log scaled in both and the warning is a false flag

In [ ]:
import scipy.sparse as sp

# check X values — should be log-normalized (small floats), not raw counts (large integers)
X_sample = cosmx_tumor_data.X[:5, :5]
if sp.issparse(X_sample):
    X_sample = X_sample.toarray()
print("CosMx X sample values:\n", X_sample)
print("CosMx X dtype:", cosmx_tumor_data.X.dtype)

X_sample_sc = sc_tumor_data.X[:5, :5]
if sp.issparse(X_sample_sc):
    X_sample_sc = X_sample_sc.toarray()
print("\nscRNA X sample values:\n", X_sample_sc)
print("scRNA X dtype:", sc_tumor_data.X.dtype)

In [ ]:

sc_rank_path = table_out / 'sc_mhc2_deg_rankings.csv'

if sc_rank_path.exists():
    sc_rank_df = pd.read_csv(sc_rank_path)
    print(f"Loaded scRNA rankings from cache ({len(sc_rank_df):,} genes)")
else:
    sc.tl.rank_genes_groups(
        sc_tumor_data,
        groupby='MHC2_clustering',
        groups=['MHC class II Low'],
        reference='MHC class II High',
        method='wilcoxon',
        key_added='rank_genes_groups',
    )
    sc_rank_df = sc.get.rank_genes_groups_df(
        sc_tumor_data, group='MHC class II Low', key='rank_genes_groups'
    )
    if 'feature_name' in sc_tumor_data.var.columns:
        gene_to_feature = sc_tumor_data.var['feature_name'].to_dict()
        sc_rank_df['gene_label'] = sc_rank_df['names'].map(gene_to_feature).fillna(sc_rank_df['names'])
    else:
        sc_rank_df['gene_label'] = sc_rank_df['names']

    sc_rank_df = sc_rank_df.sort_values('scores', ascending=False).reset_index(drop=True)
    sc_rank_df.to_csv(sc_rank_path, index=False)
    print(f"Computed and saved scRNA rankings ({len(sc_rank_df):,} genes)")

## Save ranking tables

Full ranked gene lists for both modalities saved as supplementary tables.

In [ ]:
cosmx_rank_df.to_csv(table_out / 'cosmx_mhc2_deg_rankings.csv', index=False)
sc_rank_df.to_csv(table_out / 'sc_mhc2_deg_rankings.csv', index=False)
print("Saved ranking tables to", table_out)

## Shared gene identification

For each percentile cutoff (0.5%, 1%, 2.5%, 5%), the top-scoring genes are
identified in each modality. Genes appearing in both are assigned their most
stringent shared tier. Rank metadata and tier annotations are added to both
ranking DataFrames for plotting.

In [ ]:
percentiles = [0.5, 1, 2.5, 5]

shape_map = {0.5: "o", 1: "s", 2.5: "^", 5: "X"}
size_map  = {0.5: 90,  1: 70,  2.5: 55,  5: 40}

def top_percent(df, p):
    """Return set of gene_labels in the top p% by score."""
    n = max(1, int(len(df) * p / 100))
    return set(df.nlargest(n, 'scores')['gene_label'])

def build_overlap_dict(df_a, df_b):
    """
    Find genes in the top p% of both datasets for each percentile.
    Returns dict mapping gene_label -> most stringent shared tier.
    """
    overlap = {}
    for p in percentiles:
        shared = top_percent(df_a, p) & top_percent(df_b, p)
        for g in shared:
            overlap.setdefault(g, p)  # keep smallest (most stringent) p
    return overlap

def annotate_ranks(df, overlap_dict, group_color):
    """Add rank, shared_tier, color, size, and shape columns to ranking df."""
    df = df.copy()
    df['rank']       = np.arange(1, len(df) + 1)
    df['shared_tier'] = df['gene_label'].map(overlap_dict)
    df['color']      = np.where(df['shared_tier'].notna(), group_color, 'lightgray')
    df['size']       = df['shared_tier'].map(size_map).fillna(10)
    df['shape']      = df['shared_tier'].map(shape_map).fillna('o')
    return df

## Rank-scatter plot function

Each panel plots enrichment score (x, log scale) vs rank (y, log scale, inverted)
for the top 25% most enriched genes. Non-shared genes are shown in light gray;
shared genes are colored and shaped by their most stringent concordance tier.
CosMx panels use hollow markers; scRNA panels use filled markers.

In [ ]:
def plot_rank_scatter(cosmx_df, sc_df, group_color, fig_path, top_pct=None):
    """
    Side-by-side rank-scatter plot for CosMx (hollow) and scRNA (filled).
    Shared genes highlighted by tier-specific shape and color.

    Parameters
    ----------
    cosmx_df : pd.DataFrame
        Annotated CosMx ranking df (output of annotate_ranks).
    sc_df : pd.DataFrame
        Annotated scRNA ranking df (output of annotate_ranks).
    group_color : str
        Hex color for shared gene markers.
    fig_path : Path
        Output PDF path.
    top_pct : float or None
        If provided, plot only the top X% of genes by score (e.g. 25 for top 25%).
        If None, plot all genes.
    """
    sns.set(font_scale=1.0)
    sns.set_style('ticks')

    def subset_df(df):
        if top_pct is not None:
            n_top = int(len(df) * top_pct / 100)
            return df.nlargest(n_top, 'scores').copy()
        return df.copy()

    cosmx_plot = subset_df(cosmx_df)
    sc_plot    = subset_df(sc_df)

    fig, axes = plt.subplots(
        1, 2, figsize=(8, 9),
        sharex=False, sharey=False,
        gridspec_kw={'wspace': 0.5}
    )
    fig.subplots_adjust(left=0.12, right=0.95, bottom=0.12)

    panels = [
        (cosmx_plot, 'COSMX (hollow)', 'none'),
        (sc_plot,    'scRNA-seq (filled)', 'full'),
    ]

    for ax, (df, title, fillstyle) in zip(axes, panels):
        full_n    = len(cosmx_df) if 'COSMX' in title else len(sc_df)
        plot_n    = len(cosmx_plot) if 'COSMX' in title else len(sc_plot)

        # non-shared genes
        non = df[df['shared_tier'].isna()]
        ax.scatter(
            non['scores'], non['rank'],
            s=non['size'],
            facecolors='none' if fillstyle == 'none' else 'lightgray',
            edgecolors='lightgray', alpha=0.5, linewidths=0.5,
        )

        # shared genes, per tier
        for p in percentiles:
            sub = df[df['shared_tier'] == p]
            if not sub.empty:
                ax.scatter(
                    sub['scores'], sub['rank'],
                    s=sub['size'],
                    marker=shape_map[p],
                    facecolors=sub['color'] if fillstyle == 'full' else 'none',
                    edgecolors=sub['color'],
                    alpha=0.9, linewidths=0.8,
                )

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.invert_yaxis()
        ax.set_xlim(1e-2, 1e3)
        ax.set_xticks([1e-2, 1e0, 1e2])
        ax.get_xaxis().set_major_formatter(
            plt.FuncFormatter(lambda x, _: f'{x:g}')
        )
        ax.grid(True, which='major', axis='x', color='lightgray', alpha=0.3)

        # ylim — bottom is the number of plotted genes, top gives headroom above rank=1
        ax.set_ylim(bottom=plot_n * 1.1, top=0.5)

        # percentile reference lines — based on full dataset size
        for p in percentiles:
            cutoff_rank = int(full_n * p / 100)
            ax.axhline(cutoff_rank, color='gray', linestyle='--', lw=1, alpha=0.3)
            ax.text(
                1.2e-2, cutoff_rank,
                f'top {p}%', color='gray', fontsize=10,
                va='center', ha='left',
            )

        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Enrichment score (log scale)', fontsize=11)
        ax.set_ylabel('Rank (1 = highest)', fontsize=11)
        ax.tick_params(labelsize=10)
        sns.despine(ax=ax)

    legend_elements = [
        Line2D([], [], marker=shape_map[p], color=group_color,
               label=f'Shared ≤ {p}%', markerfacecolor='none',
               markersize=8, lw=0)
        for p in percentiles
    ]
    legend_elements.append(
        Line2D([], [], marker='o', color='lightgray', label='Not shared',
               markerfacecolor='none', markersize=6, lw=0)
    )
    fig.legend(
        handles=legend_elements,
        loc='lower center',
        ncol=len(percentiles) + 1,
        frameon=False,
        fontsize=9,
        bbox_to_anchor=(0.5, -0.02),
    )

    sns.set(font_scale=1.8)
    sns.set_style('ticks')
    plt.savefig(fig_path, bbox_inches='tight')
    return fig

## Figure 3A — MHC II–low concordance

Genes enriched in MHC II–low malignant cells, ranked by Wilcoxon score.
Purple markers indicate genes concordantly enriched in the top percentile
tier across both CosMx and scRNA-seq.

### a supplemental with only the top 25%

In [ ]:
overlap_low = build_overlap_dict(cosmx_rank_df, sc_rank_df)
cosmx_low = annotate_ranks(cosmx_rank_df, overlap_low, group_color=cmap_high_low[1])
sc_low    = annotate_ranks(sc_rank_df,    overlap_low, group_color=cmap_high_low[1])

fig = plot_rank_scatter(
    cosmx_low, sc_low,
    group_color=cmap_high_low[1],
    fig_path=fig_out / 'fig3a_deg_rank_mhc2_low.pdf',
)
plt.show()

In [ ]:
fig = plot_rank_scatter(
    cosmx_low, sc_low,
    group_color=cmap_high_low[1],
    fig_path=supp_out / 'figS3a_deg_rank_mhc2_low_top25.pdf',
    top_pct=25,
)
plt.show()

## Figure 3B — MHC II–high concordance

Score direction is flipped so that positive scores now represent enrichment
in MHC II–high malignant cells. Genes are re-ranked accordingly. Orange
markers indicate concordantly enriched genes across both modalities.

In [ ]:
# flip scores so positive = enriched in MHC II High, then re-rank
cosmx_rank_high = cosmx_rank_df.copy()
sc_rank_high    = sc_rank_df.copy()

cosmx_rank_high['scores'] *= -1
sc_rank_high['scores']    *= -1

cosmx_rank_high = cosmx_rank_high.sort_values('scores', ascending=False).reset_index(drop=True)
sc_rank_high    = sc_rank_high.sort_values('scores', ascending=False).reset_index(drop=True)

overlap_high = build_overlap_dict(cosmx_rank_high, sc_rank_high)

cosmx_high = annotate_ranks(cosmx_rank_high, overlap_high, group_color=cmap_high_low[0])
sc_high    = annotate_ranks(sc_rank_high,    overlap_high, group_color=cmap_high_low[0])

fig = plot_rank_scatter(
    cosmx_high, sc_high,
    group_color=cmap_high_low[0],
    fig_path=fig_out / 'fig3b_deg_rank_mhc2_high.pdf',
)
plt.show()

In [ ]:
fig = plot_rank_scatter(
    cosmx_high, sc_high,
    group_color=cmap_high_low[1],
    fig_path=fig_out / 'fig3b_deg_rank_mhc2_high_top25.pdf',
    top_pct=25,
)
plt.show()


## Shared gene lists

Concordantly enriched genes at each percentile tier, printed for both
MHC II–low and MHC II–high comparisons.

In [ ]:
for label, overlap in [('MHC II Low', overlap_low), ('MHC II High', overlap_high)]:
    print(f"\n=== Shared genes — {label} ===")
    for p in percentiles:
        # genes at exactly this tier (most stringent assignment)
        at_tier = sorted(g for g, tier in overlap.items() if tier == p)
        # genes shared at this tier or more stringent (cumulative)
        cumulative = sorted(g for g, tier in overlap.items() if tier <= p)
        print(f"\n  Top {p}% — {len(cumulative)} genes ({len(at_tier)} first appearing at this tier):")
        print(f"  {', '.join(cumulative) if cumulative else 'None'}")